# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the default model directory.
model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/diffuser-patrolling-obs200/wandb/run-20260316_170952-7dqf5om8/files"

# Set the output file.
# output_file = "output.zarr"

# Set the output file automatically.
output_file = "./patrolling_cumberland_" + model_dir.split("/")[-2] + ".zarr"
# output_file = "./patrolling_" + model_dir.split("/")[-2] + ".zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
import time
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [3]:
def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            v = value["value"]
            if v == None:
                continue
            if type(v) == bool:
                if v:
                    arglist.append(f"--{key}")
                else:
                    arglist.append(f"--no-{key}")
            elif type(v) == list:
                if len(v) == 0:
                    continue
                arglist.append(f"--{key}")
                arglist.extend([str(i) for i in v])
            else:
                arglist.append(f"--{key}")
                arglist.append(str(v))
    return arglist


def wrap_arg_val(val):
    return {"value": val}


def load_args(model_dir):
    # Load arguments from the config file.
    config_file = os.path.join(model_dir, "config.yaml")
    args = yaml.load(open(config_file), Loader=yaml.FullLoader)

    # Set required eval-specific arguments. Do not change these!
    args["use_wandb"] = wrap_arg_val(False)
    args["use_render"] = wrap_arg_val(False)
    args["use_eval"] = wrap_arg_val(True)
    args["model_dir"] = wrap_arg_val(model_dir)
    args["eval_output_file"] = wrap_arg_val(output_file)


    # Set default arguments for evaluation. Feel free to change these!
    args["cuda"] = wrap_arg_val(True)
    args["cuda_idx"] = wrap_arg_val(3)
    args["cuda_idx_predictor"] = wrap_arg_val(3)
    args["eval_episodes"] = wrap_arg_val(3)
    args["episode_length"] = wrap_arg_val(200)
    args["seed"] = wrap_arg_val(1)

    args["observation_probability"] = wrap_arg_val(0.5)
    args["observation_mask"] = wrap_arg_val(True)

    args["graph_name"] = wrap_arg_val("cumberland")
    args["graph_file"] = wrap_arg_val("../patrolling_zoo/graphs/cumberland.graph")

    return args

# Load arguments for the default model.
args = load_args(model_dir)

# Evaluation

In [4]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)

    # obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    # obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    comms_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    obs_radii = [50.0, 100.0, 200.0]

    for comms_prob in comms_probabilities:
        for obs_radius in obs_radii:
            args = load_args(model_dir)

            series_name = f"obs{obs_radius} - {args['experiment_name']['value']}"

            args["eval_series"] = wrap_arg_val(f"AR0 - {series_name}")
            args["experiment_name"] = wrap_arg_val(f"comms_prob_{comms_prob:.2f}")
            args["communication_probability"] = wrap_arg_val(comms_prob)
            args["diffusion_autoregression_steps"] = wrap_arg_val(0)
            args["observation_radius"] = wrap_arg_val(obs_radius)
            start_process(args)
            
            argscopy = deepcopy(args)
            argscopy["eval_series"] = wrap_arg_val(f"AR1 - {series_name}")
            argscopy["diffusion_autoregression_steps"] = wrap_arg_val(1)
            start_process(argscopy)

            # argscopy = deepcopy(args)
            # argscopy["eval_series"] = wrap_arg_val(f"AR7 - {series_name}")
            # argscopy["diffusion_autoregression_steps"] = wrap_arg_val(7)
            # start_process(argscopy)

            # VARIANT 2: No prediction.
            argscopy = deepcopy(args)
            argscopy["eval_series"] = wrap_arg_val(f"no-prediction - {series_name}")
            argscopy["prediction_disable"] = wrap_arg_val(True)
            start_process(argscopy)


    # VARIANT 3: Load an entirely different model! (example)
    # argsnew = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/diffuser-patrolling-obs200/wandb/run-20250731_182446-4h2njfwx/files")
    # argsnew["experiment_name"] = wrap_arg_val("diffusion-prediction-obs200")
    # start_process(argsnew)


    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    MAX_PROCESSES = 30
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


/data/anthony/dev/qmas_task_allocation-1/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '66', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy', '--observation_probability', '0.5']
  warnings.warn(f"Unknown arguments: {unknown_args}")


Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.
u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False


/data/anthony/dev/qmas_task_allocation-1/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '66', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy', '--observation_probability', '0.5']
  warnings.warn(f"Unknown arguments: {unknown_args}")




Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False

...

u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


/data/anthony/dev/qmas_task_allocation-1/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '66', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy', '--observation_probability', '0.5']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 10
Pettingzoo arguments validated: base
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_env
Pettingzoo arguments validated: base
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_envPettingzooEnv: Attempting to pass unpacked argparse namespace to environment.

PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': -1, 'num_agents': 4, 'alpha': 1.0, 'beta': 0.5, 'graph_file': '../patrolling_zoo/graphs/cumberland.graph', 'graph_random

/home/anthony/miniconda3/envs/qmas2/lib/python3.10/site-packages/torchrl/__init__.py:43: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas2/lib/python3.10/site-packages/torchrl/__init__.py:43: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


R_Critic: Use CNN: False, Use MLP: True
R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 1653.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 1653.
Initialized QmasAlgorithm with 1 predictors. Threaded training: True
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 10
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 10
Buffer initialized with episode length 200
R_Critic: Use CNN: False, Use MLP: True
R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 1653.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 16

# Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='AR0 - obs100.0 - diffuser-patrollin…

In [6]:
import plotly.graph_objects as go
import numpy as np

# Data for plotting
categories = {
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_trajectory_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['avg_idleness']) for experiment in output[series]]
                for series in output
        }
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['avg_idleness'][:, -1]) for experiment in output[series]]
                for series in output
        }
    },
    'avg_idleness_over_time': {
        'title': 'Avg. Idleness vs. Time',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per test/experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                experiment: np.mean(output[series][experiment]['avg_idleness'], axis=0)
                for experiment in output[series]
            }
            for series in output
        }
    },
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

for category in categories:
    fig = go.Figure()
    x = categories[category].get('x', None)

    if category == 'avg_idleness_over_time':
        for series, tests in categories[category]['y'].items():
            for test_name, values in tests.items():
                x_local = np.arange(1, len(values) + 1)
                fig.add_trace(go.Scatter(
                    x=x_local,
                    y=values,
                    mode='lines',
                    name=f"{series} | {test_name}",
                    hovertemplate=f'{series} | {test_name}' + ': %{y:.4f}<extra></extra>',
                    line={
                        "color": 'red' if "no-prediction" in series else 'blue',
                    }
                ))
    elif type(categories[category]['y']) == dict:
        for series, values in categories[category]['y'].items():
            x_local = x
            if x_local is None:
                x_local = np.linspace(0, 1, len(values))
            fig.add_trace(go.Scatter(
                x=x_local,
                y=values,
                mode='lines+markers',
                name=series,
                hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
            ))
    else:
        values = categories[category]['y']
        x_local = x
        if x_local is None:
            x_local = np.linspace(0, 1, len(values))
        fig.add_trace(go.Scatter(
            x=x_local,
            y=values,
            mode='lines+markers',
            hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
        ))
    
    title = categories[category]['title']
    xlabel = 'Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel']
    ylabel = f"Timesteps out of {timesteps}" if "ylabel" not in categories[category] else categories[category]['ylabel']
    
    fig.update_layout(
        # title=f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}",
        title=f"{title}",
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        hovermode='closest',
        showlegend=True,
        autosize=True,
        height=600,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(
            orientation='h',
            yanchor='top',
            y=-0.2,
            xanchor='center',
            x=0.5
        ),
    )
    
    fig.show()